In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# 연산자 클래스 개요


<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하시기를 권장합니다.

```
qiskit[all]~=2.3.0
```
</details>
Qiskit에서 양자 연산자는 [`quantum_info`](https://docs.quantum.ibm.com/api/qiskit/quantum_info) 모듈의 클래스를 사용하여 표현됩니다. 가장 중요한 연산자 클래스는 [`SparsePauliOp`](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp)로, 일반적인 양자 연산자를 파울리 문자열의 선형 결합으로 표현합니다. `SparsePauliOp`는 양자 관측량을 표현하는 데 가장 일반적으로 사용되는 클래스입니다. 이 페이지의 나머지 부분에서는 `SparsePauliOp` 및 기타 연산자 클래스의 사용 방법을 설명합니다.

In [1]:
import numpy as np
from qiskit.quantum_info.operators import Operator, Pauli, SparsePauliOp

## SparsePauliOp
[`SparsePauliOp`](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp) 클래스는 파울리 문자열의 선형 결합을 표현합니다. `SparsePauliOp`를 초기화하는 방법은 여러 가지가 있지만, 가장 유연한 방법은 다음 코드 셀에서 시연하는 것처럼 [`from_sparse_list`](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp#from_sparse_list) 메서드를 사용하는 것입니다. `from_sparse_list`는 `(pauli_string, qubit_indices, coefficient)` 삼중항의 목록을 받습니다.

In [2]:
op1 = SparsePauliOp.from_sparse_list(
    [("ZX", [1, 4], 1.0), ("YY", [0, 3], -1 + 1j)], num_qubits=5
)
op1

SparsePauliOp(['XIIZI', 'IYIIY'],
              coeffs=[ 1.+0.j, -1.+1.j])

`SparsePauliOp` supports arithmetic operations, as demonstrated in the following code cell.

In [3]:
op2 = SparsePauliOp.from_sparse_list(
    [("XXZ", [0, 1, 4], 1 + 2j), ("ZZ", [1, 2], -1 + 1j)], num_qubits=5
)

# Addition
print("op1 + op2:")
print(op1 + op2)
print()
# Multiplication by a scalar
print("2 * op1:")
print(2 * op1)
print()
# Operator multiplication (composition)
print("op1 @ op2:")
print(op1 @ op2)
print()
# Tensor product
print("op1.tensor(op2):")
print(op1.tensor(op2))

op1 + op2:
SparsePauliOp(['XIIZI', 'IYIIY', 'ZIIXX', 'IIZZI'],
              coeffs=[ 1.+0.j, -1.+1.j,  1.+2.j, -1.+1.j])

2 * op1:
SparsePauliOp(['XIIZI', 'IYIIY'],
              coeffs=[ 2.+0.j, -2.+2.j])

op1 @ op2:
SparsePauliOp(['YIIYX', 'XIZII', 'ZYIXZ', 'IYZZY'],
              coeffs=[ 1.+2.j, -1.+1.j, -1.+3.j,  0.-2.j])

op1.tensor(op2):
SparsePauliOp(['XIIZIZIIXX', 'XIIZIIIZZI', 'IYIIYZIIXX', 'IYIIYIIZZI'],
              coeffs=[ 1.+2.j, -1.+1.j, -3.-1.j,  0.-2.j])


`SparsePauliOp`는 다음 코드 셀에서 시연하는 것처럼 산술 연산을 지원합니다.

In [4]:
op1 = Pauli("iXX")
op1

Pauli('iXX')

The following code cell demonstrates the use of some attributes and methods.

In [5]:
print(f"Dimension of {op1}: {op1.dim}")
print(f"Phase of {op1}: {op1.phase}")
print(f"Matrix representation of {op1}: \n {op1.to_matrix()}")

Dimension of iXX: (4, 4)
Phase of iXX: 3
Matrix representation of iXX: 
 [[0.+0.j 0.+0.j 0.+0.j 0.+1.j]
 [0.+0.j 0.+0.j 0.+1.j 0.+0.j]
 [0.+0.j 0.+1.j 0.+0.j 0.+0.j]
 [0.+1.j 0.+0.j 0.+0.j 0.+0.j]]


## Pauli
[`Pauli`](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Pauli) 클래스는 집합 $\set{+1, i, -1, -i}$에서 선택된 선택적 위상 계수를 가진 단일 파울리 문자열을 표현합니다. `Pauli`는 `{"I", "X", "Y", "Z"}` 집합의 문자로 구성된 문자열을 전달하여 초기화할 수 있으며, 선택적으로 위상 계수를 표현하기 위해 `{"", "i", "-", "-i"}` 중 하나를 접두사로 붙일 수 있습니다.

In [6]:
XX = Operator(
    np.array(
        [
            [0, 0, 0, 1],
            [0, 0, 1, 0],
            [0, 1, 0, 0],
            [1, 0, 0, 0],
        ]
    )
)
XX

Operator([[0.+0.j, 0.+0.j, 0.+0.j, 1.+0.j],
          [0.+0.j, 0.+0.j, 1.+0.j, 0.+0.j],
          [0.+0.j, 1.+0.j, 0.+0.j, 0.+0.j],
          [1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j]],
         input_dims=(2, 2), output_dims=(2, 2))


The operator object stores the underlying matrix, and the input and output dimension of subsystems.

* `data`: To access the underlying Numpy array, you can use the `Operator.data` property.
* `dims`: To return the total input and output dimension of the operator, you can use the `Operator.dim` property. *Note: the output is returned as a tuple* `(input_dim, output_dim)`, *which is the reverse of the shape of the underlying matrix.*

In [7]:
XX.data

array([[0.+0.j, 0.+0.j, 0.+0.j, 1.+0.j],
       [0.+0.j, 0.+0.j, 1.+0.j, 0.+0.j],
       [0.+0.j, 1.+0.j, 0.+0.j, 0.+0.j],
       [1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j]])

In [8]:
input_dim, output_dim = XX.dim
input_dim, output_dim

(4, 4)

The operator class also keeps track of subsystem dimensions, which can be used for composing operators together. These can be accessed using the `input_dims` and `output_dims` functions.

For $2^N$ by $2^M$ operators, the input and output dimensions are automatically assumed to be M-qubit and N-qubit:

In [9]:
op = Operator(np.random.rand(2**1, 2**2))
print("Input dimensions:", op.input_dims())
print("Output dimensions:", op.output_dims())

Input dimensions: (2, 2)
Output dimensions: (2,)


`Pauli` 객체는 수반(adjoint) 결정, 다른 `Pauli`와의 (반)교환 여부 확인, 다른 `Pauli`와의 내적 계산 등 연산자를 조작하는 다양한 메서드를 보유하고 있습니다. 자세한 내용은 [API 문서](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Pauli)를 참고하세요.
## Operator
[`Operator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Operator) 클래스는 일반적인 선형 연산자를 표현합니다. `SparsePauliOp`와 달리, `Operator`는 선형 연산자를 밀집 행렬로 저장합니다. 밀집 행렬을 저장하는 데 필요한 메모리는 Qubit 수에 따라 지수적으로 증가하기 때문에, `Operator` 클래스는 소수의 Qubit에만 사용하기에 적합합니다.

Numpy 배열을 직접 전달하여 연산자의 행렬을 저장하는 방식으로 `Operator`를 초기화할 수 있습니다. 예를 들어, 다음 코드 셀은 2-Qubit 파울리 XX 연산자를 생성합니다:

In [10]:
op = Operator(np.random.rand(6, 6))
print("Input dimensions:", op.input_dims())
print("Output dimensions:", op.output_dims())

Input dimensions: (6,)
Output dimensions: (6,)


The input and output dimension can also be manually specified when initializing a new operator:

In [11]:
# Force input dimension to be (4,) rather than (2, 2)
op = Operator(np.random.rand(2**1, 2**2), input_dims=[4])
print("Input dimensions:", op.input_dims())
print("Output dimensions:", op.output_dims())

Input dimensions: (4,)
Output dimensions: (2,)


In [12]:
# Specify system is a qubit and qutrit
op = Operator(np.random.rand(6, 6), input_dims=[2, 3], output_dims=[2, 3])
print("Input dimensions:", op.input_dims())
print("Output dimensions:", op.output_dims())

Input dimensions: (2, 3)
Output dimensions: (2, 3)


You can also extract just the input or output dimensions of a subset of subsystems using the `input_dims` and `output_dims` functions:

In [13]:
print("Dimension of input system 0:", op.input_dims([0]))
print("Dimension of input system 1:", op.input_dims([1]))

Dimension of input system 0: (2,)
Dimension of input system 1: (3,)


## Next steps

<Admonition type="tip" title="Recommendations">
  - Learn how to [specify observables in the Pauli basis](./specify-observables-pauli).
  - See an example of using operators in the [Combine error mitigation options with the Estimator primitive](/docs/tutorials/combine-error-mitigation-techniques) tutorial.
  - Read more [in-depth coverage of the Operator class](./operator-class).
  - Explore the [Operator API](/docs/api/qiskit/qiskit.quantum_info.Operator#operator) reference.
</Admonition>